In [ ]:
# Import necessary libraries
import pandas as pd

# Load the datasets
train_data = pd.read_csv('final data/expanded_train_data.csv')
validation_data = pd.read_csv('final data/expanded_validation_data.csv')
test_data = pd.read_csv('final data/expanded_test_data.csv')

# Load the models' predictions
gpt4o_results = pd.read_csv('cte/gpt4o/results/gpt4o_results.csv')
syntactic_parsing_results = pd.read_csv('cte/rule-based syntactic parsing/results/syntactic_parsing_results.csv')
bert_results = pd.read_csv('cte/bert/results/bert_results.csv')


def form_triples(data):
    """
    Function to form subject-predicate-object (SPO) triples from token-level labels in the dataset.
    
    """
    # Group data by sentences to process each sentence individually
    grouped = data.groupby('sentence_id')

    # Prepare a list to store the triples
    triples = []

    for sentence_id, group in grouped:
        # Extract all tokens by their labels
        subject_tokens = ' '.join(group[group['spo_label'] == 'Subject']['token'])
        predicate_tokens = ' '.join(group[group['spo_label'] == 'Predicate']['token'])
        object_tokens = ' '.join(group[group['spo_label'] == 'Object']['token'])

        # Form the triple and add it to the list
        triple = (subject_tokens, predicate_tokens, object_tokens)
        triples.extend([triple] * len(group))  # Extend the triple for each token in the sentence

    # Add the triples as a new column to the original dataframe
    data['triple'] = triples


def form_triples_bert(data):
    """
    Function to form subject-predicate-object (SPO) triples from token-level predicted and gold labels in the dataset.
    
    """
    # Group data by sentences to process each sentence individually
    grouped = data.groupby('sentence_id')

    # Prepare a list to store the triples
    predicted_triples = []
    gold_triples = []

    for sentence_id, group in grouped:
        # Extract tokens by their predicted labels
        predicted_subject_tokens = ' '.join(group[group['predicted_label'] == 'Subject']['token'])
        predicted_predicate_tokens = ' '.join(group[group['predicted_label'] == 'Predicate']['token'])
        predicted_object_tokens = ' '.join(group[group['predicted_label'] == 'Object']['token'])

        # Extract tokens by their gold labels
        gold_subject_tokens = ' '.join(group[group['gold_label'] == 'Subject']['token'])
        gold_predicate_tokens = ' '.join(group[group['gold_label'] == 'Predicate']['token'])
        gold_object_tokens = ' '.join(group[group['gold_label'] == 'Object']['token'])

        # Form the triples
        predicted_triple = (predicted_subject_tokens, predicted_predicate_tokens, predicted_object_tokens)
        gold_triple = (gold_subject_tokens, gold_predicate_tokens, gold_object_tokens)

        # Extend the triples for each token in the sentence
        predicted_triples.extend([predicted_triple] * len(group))
        gold_triples.extend([gold_triple] * len(group))

    # Add the triples as new columns to the original dataframe
    data['predicted_triple'] = predicted_triples
    data['gold_triple'] = gold_triples


# Apply the function to all datasets
form_triples(train_data)
form_triples(validation_data)
form_triples(test_data)

# Apply the function to models' predictions
form_triples(gpt4o_results)
form_triples(syntactic_parsing_results)
form_triples_bert(bert_results)

# Save the new datasets
train_data.to_csv('final data/training_dataset_triples.csv', index=False)
validation_data.to_csv('final data/validation_dataset_triples.csv', index=False)
test_data.to_csv('final data/testing_dataset_triples.csv', index=False)

# Save the new predictions
gpt4o_results.to_csv('cte/gpt4o/results/gpt4o_results_triples.csv', index=False)
syntactic_parsing_results.to_csv('cte/rule-based syntactic parsing/results/syntactic_parsing_results_triples.csv', index=False)
bert_results.to_csv('cte/bert/results/bert_results_triples.csv', index=False)
